# Tempo de UTI em insuficiência cardíaca: estadia inteira e AUC de "estadia longa"

**O que este caderno faz**

1. Prevê `los_dias` (dias de UTI) com sinais vitais e exames de **toda a estadia**, não só das primeiras 24 horas.
2. Mostra MAE, RMSE e R² no teste, com divisão por paciente e `random_state=42`.
3. Converte as previsões em "estadia longa" (mais de 3,55 e mais de 7 dias) e calcula a AUC.

**Regras que valem aqui**

- O CID só define a coorte. Ele não entra como variável e não é o alvo.
- O alvo é `los_dias` e fica fora de X. Contagem de registros e horário da alta também ficam fora.
- Imputação, one-hot e escala são ajustados só no treino.

**Atenção:** com a estadia inteira, o modelo vê ureia, creatinina e SpO2 mínimas e máximas que só existem depois da alta. Serve para entender o que acompanha a permanência, não para prever na entrada da UTI.

**Tempo:** a primeira execução lê `chartevents` e `labevents` e leva cerca de 40 minutos. O resultado é salvo em `artefatos_los_estadia/tabela_estadia_completa.csv`. Nas próximas vezes o caderno lê esse arquivo e roda em poucos minutos.

Pré-requisito: ter rodado o pré-processamento, que gera `artefatos/coorte_bruta.csv`.

## 1. Configuração

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from IPython.display import display
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, Ridge
from sklearn.metrics import (
    balanced_accuracy_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, GroupKFold, GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBRegressor

RANDOM_STATE = 42
RAIZ = Path.cwd()
MIMIC = RAIZ / "data" / "mimic-iv"
COORTE = RAIZ / "artefatos" / "coorte_bruta.csv"
SAIDA = RAIZ / "artefatos_los_estadia"
SAIDA.mkdir(exist_ok=True)
CACHE = SAIDA / "tabela_estadia_completa.csv"
PREVISOES_24H = RAIZ / "artefatos" / "previsoes_teste.csv"

LOS_MAXIMO_TIPICO = 14.0
CORTES_DIAS = [3.55, 7.0]
ALPHAS = np.logspace(-3, 3, 13)
N_SHAP = 1500

# itemid do MIMIC-IV: chartevents (sinais vitais) e labevents (exames)
ITEM_FC, ITEM_FR, ITEM_SPO2 = 220045, 220210, 220277
ITEM_PAM_INV, ITEM_PAM_NI = 220052, 220181
ITEM_CREAT, ITEM_UREIA, ITEM_NA = 50912, 51006, 50983
ITEM_K, ITEM_GLIC, ITEM_PTT = 50971, 50931, 51275
ITENS_VITAIS = {ITEM_FC, ITEM_FR, ITEM_SPO2, ITEM_PAM_INV, ITEM_PAM_NI}
ITENS_LABS = {ITEM_CREAT, ITEM_UREIA, ITEM_NA, ITEM_K, ITEM_GLIC, ITEM_PTT}

# Fora destas faixas o valor é tratado como erro de registro (ausente).
FAIXAS = {
    ITEM_FC: (20.0, 280.0),
    ITEM_FR: (4.0, 70.0),
    ITEM_SPO2: (50.0, 100.0),
    ITEM_PAM_INV: (20.0, 250.0),
    ITEM_PAM_NI: (20.0, 250.0),
    ITEM_CREAT: (0.1, 30.0),
    ITEM_UREIA: (1.0, 300.0),
    ITEM_NA: (100.0, 180.0),
    ITEM_K: (1.5, 9.0),
    ITEM_GLIC: (20.0, 1500.0),
    ITEM_PTT: (10.0, 200.0),
}
NOME = {
    ITEM_FC: "frequencia_cardiaca",
    ITEM_FR: "frequencia_respiratoria",
    ITEM_SPO2: "spo2",
    ITEM_PAM_INV: "pam_invasiva",
    ITEM_PAM_NI: "pam_nao_invasiva",
    ITEM_CREAT: "creatinina",
    ITEM_UREIA: "ureia",
    ITEM_NA: "sodio",
    ITEM_K: "potassio",
    ITEM_GLIC: "glicose",
    ITEM_PTT: "ptt",
}
PAINEL = [
    "idade", "feve", "pam", "frequencia_cardiaca", "frequencia_respiratoria",
    "spo2", "creatinina", "ureia", "sodio", "potassio", "glicose", "ptt",
]
EXTREMOS = ["pam", "spo2", "creatinina", "ureia"]
CATEGORICAS = ["genero", "tipo_admissao", "unidade"]

print("Coorte:", COORTE, "| existe:", COORTE.exists())
print("Cache da tabela:", CACHE, "| existe:", CACHE.exists())

## 2. Leitura dos sinais vitais e dos exames da estadia

Cada sinal vale entre a entrada (`intime`) e a saída (`outtime`) da UTI. Para cada estadia ficam a mediana, o mínimo e o máximo.

In [ ]:
def metricas(y, previsto):
    return {
        "MAE": float(mean_absolute_error(y, previsto)),
        "RMSE": float(np.sqrt(mean_squared_error(y, previsto))),
        "R2": float(r2_score(y, previsto)),
    }


def achar(nome):
    for caminho in (MIMIC / nome, MIMIC / f"{nome}.gz", MIMIC / "icu" / nome, MIMIC / "hosp" / nome):
        if caminho.exists():
            return caminho
    raise FileNotFoundError(nome)


def aplicar_faixas(item, valor):
    saida = pd.Series(np.nan, index=valor.index, dtype="float64")
    for codigo, (baixo, alto) in FAIXAS.items():
        mascara = item == codigo
        bruto = valor.loc[mascara]
        saida.loc[mascara] = bruto.where((bruto >= baixo) & (bruto <= alto))
    return saida


def varrer(caminho, colunas, itens, ids, coluna_id):
    partes = []
    lidas = 0
    for bloco in pd.read_csv(caminho, usecols=colunas, chunksize=1_000_000, low_memory=False):
        lidas += len(bloco)
        item = pd.to_numeric(bloco["itemid"], errors="coerce")
        identificador = pd.to_numeric(bloco[coluna_id], errors="coerce")
        valor = pd.to_numeric(bloco["valuenum"], errors="coerce")
        ficar = item.isin(itens) & identificador.isin(ids) & valor.notna()
        if ficar.any():
            recorte = pd.DataFrame({
                coluna_id: identificador.loc[ficar].astype(np.int64).to_numpy(),
                "charttime": pd.to_datetime(bloco.loc[ficar, "charttime"], errors="coerce"),
                "itemid": item.loc[ficar].astype(np.int64).to_numpy(),
                "valuenum": aplicar_faixas(item.loc[ficar], valor.loc[ficar]).to_numpy(),
            })
            partes.append(recorte.dropna(subset=["charttime", "valuenum"]))
        if lidas % 20_000_000 == 0:
            print(f"  {caminho.name}: {lidas:,} linhas lidas", flush=True)
    return pd.concat(partes, ignore_index=True)


def resumir(medidas, janela, coluna_id):
    cruzado = medidas.merge(janela, on=coluna_id, how="inner")
    cruzado = cruzado[(cruzado["charttime"] >= cruzado["intime"]) & (cruzado["charttime"] < cruzado["outtime"])]
    cruzado["exame"] = cruzado["itemid"].map(NOME)
    resumo = (
        cruzado.groupby(["stay_id", "exame"])["valuenum"]
        .agg(mediana="median", minimo="min", maximo="max")
        .reset_index()
    )
    mediana = resumo.pivot(index="stay_id", columns="exame", values="mediana")
    minimo = resumo.pivot(index="stay_id", columns="exame", values="minimo").add_suffix("_min")
    maximo = resumo.pivot(index="stay_id", columns="exame", values="maximo").add_suffix("_max")
    return mediana.join(minimo).join(maximo)


def montar_tabela():
    coorte = pd.read_csv(COORTE)
    estadias = pd.read_csv(achar("icustays.csv"), usecols=["stay_id", "intime", "outtime", "first_careunit"])
    estadias["intime"] = pd.to_datetime(estadias["intime"])
    estadias["outtime"] = pd.to_datetime(estadias["outtime"])
    tabela = coorte.merge(estadias, on="stay_id", how="inner")
    tabela["unidade"] = tabela["first_careunit"].astype("string").fillna("desconhecido")
    janela_stay = tabela[["stay_id", "intime", "outtime"]].drop_duplicates("stay_id")
    janela_pac = tabela[["subject_id", "stay_id", "intime", "outtime"]].drop_duplicates("stay_id")

    print("Lendo sinais vitais da estadia inteira...", flush=True)
    vitais = varrer(
        achar("chartevents.csv"),
        ["stay_id", "charttime", "itemid", "valuenum"],
        ITENS_VITAIS,
        set(janela_stay["stay_id"].astype(np.int64)),
        "stay_id",
    )
    vitais = resumir(vitais, janela_stay, "stay_id")

    def coluna(nome):
        return vitais[nome] if nome in vitais.columns else pd.Series(np.nan, index=vitais.index)

    vitais["pam"] = coluna("pam_invasiva").combine_first(coluna("pam_nao_invasiva"))
    vitais["pam_min"] = coluna("pam_invasiva_min").combine_first(coluna("pam_nao_invasiva_min"))
    vitais["pam_max"] = coluna("pam_invasiva_max").combine_first(coluna("pam_nao_invasiva_max"))

    print("Lendo exames da estadia inteira...", flush=True)
    labs = varrer(
        achar("labevents.csv"),
        ["subject_id", "charttime", "itemid", "valuenum"],
        ITENS_LABS,
        set(janela_pac["subject_id"].astype(np.int64)),
        "subject_id",
    )
    labs = resumir(labs, janela_pac, "subject_id")
    clinico = vitais.join(labs, how="outer")
    return tabela.merge(clinico, left_on="stay_id", right_index=True, how="left")

In [ ]:
if CACHE.exists():
    tabela = pd.read_csv(CACHE)
    print("Tabela lida do cache.")
else:
    tabela = montar_tabela()
    tabela.to_csv(CACHE, index=False, encoding="utf-8-sig")
    print("Tabela calculada e salva no cache.")

tabela = tabela.dropna(subset=["los_dias", "subject_id"]).drop_duplicates("stay_id")
tabela = tabela[tabela["los_dias"] > 0].copy()
print(f"Estadias: {len(tabela):,} | pacientes: {tabela['subject_id'].nunique():,}")
display(tabela["los_dias"].describe().round(2).to_frame("los_dias"))

## 3. Divisão por paciente, imputação, one-hot e escala

O mesmo paciente nunca aparece no treino e no teste. Mediana, categorias e escala vêm só do treino.

In [ ]:
def preparar(tabela):
    divisor = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
    treino_idx, teste_idx = next(divisor.split(tabela, groups=tabela["subject_id"]))
    treino = tabela.iloc[treino_idx].copy()
    teste = tabela.iloc[teste_idx].copy()
    assert not set(treino["subject_id"]) & set(teste["subject_id"]), "paciente repetido"

    for c in CATEGORICAS:
        treino[c] = treino[c].astype("string").fillna("desconhecido")
        teste[c] = teste[c].astype("string").fillna("desconhecido")
    try:
        encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore", dtype=np.float64)
    except TypeError:
        encoder = OneHotEncoder(sparse=False, handle_unknown="ignore", dtype=np.float64)
    nomes = None
    d_treino = encoder.fit_transform(treino[CATEGORICAS])
    nomes = encoder.get_feature_names_out(CATEGORICAS)
    d_treino = pd.DataFrame(d_treino, columns=nomes, index=treino.index)
    d_teste = pd.DataFrame(encoder.transform(teste[CATEGORICAS]), columns=nomes, index=teste.index)

    numericas = [c for c in PAINEL if c in treino.columns]
    numericas += [f"{c}_min" for c in EXTREMOS if f"{c}_min" in treino.columns]
    numericas += [f"{c}_max" for c in EXTREMOS if f"{c}_max" in treino.columns]
    imputer = SimpleImputer(strategy="median")
    n_treino = pd.DataFrame(imputer.fit_transform(treino[numericas]), columns=numericas, index=treino.index)
    n_teste = pd.DataFrame(imputer.transform(teste[numericas]), columns=numericas, index=teste.index)

    x_treino = pd.concat([n_treino, d_treino], axis=1)
    x_teste = pd.concat([n_teste, d_teste], axis=1).reindex(columns=x_treino.columns, fill_value=0.0)
    escala = StandardScaler()
    x_treino_esc, x_teste_esc = x_treino.copy(), x_teste.copy()
    x_treino_esc[numericas] = escala.fit_transform(x_treino[numericas])
    x_teste_esc[numericas] = escala.transform(x_teste[numericas])

    proibidas = [c for c in x_treino.columns if "los" in c or "icd" in c.lower() or "diag" in c.lower()]
    assert not proibidas, f"coluna proibida em X: {proibidas}"
    return treino, teste, x_treino, x_teste, x_treino_esc, x_teste_esc

## 4. Modelos

Ridge e Lasso (escala aplicada), Random Forest e XGBoost. Todos aprendem `log(1 + los_dias)`. A previsão volta para dias com `expm1` antes das métricas.

In [ ]:
def treinar(x_treino, x_treino_esc, y_log, grupos):
    cv = GroupKFold(n_splits=5)
    ridge = GridSearchCV(Ridge(), {"alpha": ALPHAS}, cv=cv, scoring="neg_mean_absolute_error", n_jobs=1)
    ridge.fit(x_treino_esc, y_log, groups=grupos)
    lasso = GridSearchCV(
        Lasso(max_iter=20000, random_state=RANDOM_STATE), {"alpha": ALPHAS},
        cv=cv, scoring="neg_mean_absolute_error", n_jobs=1,
    )
    lasso.fit(x_treino_esc, y_log, groups=grupos)
    floresta = RandomForestRegressor(
        n_estimators=300, max_depth=12, min_samples_leaf=5, max_features="sqrt",
        n_jobs=1, random_state=RANDOM_STATE,
    )
    floresta.fit(x_treino, y_log)
    xgb = XGBRegressor(
        n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
        min_child_weight=5, objective="reg:squarederror", n_jobs=1, random_state=RANDOM_STATE,
    )
    xgb.fit(x_treino, y_log)
    return {
        "Ridge": ridge.best_estimator_, "Lasso": lasso.best_estimator_,
        "Random Forest": floresta, "XGBoost": xgb,
    }


def avaliar(nome, modelos, x_teste, x_teste_esc, y_dias, y_log_treino):
    mediana = float(np.expm1(np.median(y_log_treino)))
    linhas = [{"coorte": nome, "modelo": "Mediana do treino", **metricas(y_dias, np.full(len(y_dias), mediana))}]
    entrada = {"Ridge": x_teste_esc, "Lasso": x_teste_esc, "Random Forest": x_teste, "XGBoost": x_teste}
    previsoes = {"los_dias": y_dias.to_numpy()}
    for modelo, x in entrada.items():
        previsto = np.clip(np.expm1(modelos[modelo].predict(x)), 0, None)
        linhas.append({"coorte": nome, "modelo": modelo, **metricas(y_dias, previsto)})
        previsoes[modelo] = previsto
    pd.DataFrame(previsoes).to_csv(SAIDA / f"previsoes_{nome}.csv", index=False, encoding="utf-8-sig")
    return pd.DataFrame(linhas)


def explicar(modelo, x_teste, nome):
    amostra = x_teste.sample(n=min(N_SHAP, len(x_teste)), random_state=RANDOM_STATE)
    valores = np.asarray(shap.TreeExplainer(modelo)(amostra).values)
    importancia = (
        pd.DataFrame({"variavel": amostra.columns, "shap_medio_abs": np.abs(valores).mean(axis=0)})
        .sort_values("shap_medio_abs", ascending=False)
        .reset_index(drop=True)
    )
    importancia.to_csv(SAIDA / f"shap_{nome}.csv", index=False, encoding="utf-8-sig")
    return importancia


def rodar(tabela, nome):
    treino, teste, x_treino, x_teste, x_treino_esc, x_teste_esc = preparar(tabela)
    y_log = np.log1p(treino["los_dias"].to_numpy(dtype=float))
    modelos = treinar(x_treino, x_treino_esc, y_log, treino["subject_id"])
    metricas_tabela = avaliar(nome, modelos, x_teste, x_teste_esc, teste["los_dias"], y_log)
    importancia = explicar(modelos["XGBoost"], x_teste, nome)
    print(f"{nome}: treino {len(treino):,} | teste {len(teste):,} | colunas {x_treino.shape[1]}")
    return metricas_tabela, importancia

### 4.1 Estadia inteira, todas as estadias

In [ ]:
metricas_todas, shap_todas = rodar(tabela, "estadia_completa")
display(metricas_todas.round(3))

### 4.2 Estadia inteira, só estadias de até 14 dias

Tira a cauda longa para ver quanto ela pesa no R².

In [ ]:
curta = tabela[tabela["los_dias"] <= LOS_MAXIMO_TIPICO].copy()
metricas_curta, shap_curta = rodar(curta, "ate_14_dias")
display(metricas_curta.round(3))

metricas_final = pd.concat([metricas_todas, metricas_curta], ignore_index=True)
metricas_final.to_csv(SAIDA / "metricas_teste.csv", index=False, encoding="utf-8-sig")

## 5. Que variáveis pesam mais (SHAP no XGBoost)

A escala é `log(1 + dias)`, não dias.

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(13, 5))
for eixo, (titulo, tabela_shap) in zip(eixos, [("Estadia inteira", shap_todas), ("Até 14 dias", shap_curta)]):
    topo = tabela_shap.head(12).iloc[::-1]
    eixo.barh(topo["variavel"], topo["shap_medio_abs"], color="#1f4e79")
    eixo.set_title(titulo)
    eixo.set_xlabel("SHAP médio absoluto, log(1 + dias)")
plt.tight_layout()
plt.savefig(SAIDA / "shap_estadia_completa_e_14_dias.png", dpi=150)
plt.show()

## 6. Classificação de "estadia longa" a partir das previsões em dias

Nenhum modelo é treinado de novo. A previsão em dias vira a nota do classificador. O valor real define quem é longo.

- Corte 3,55 dias: o do artigo de Yang et al. (2025).
- Corte 7 dias: critério mais exigente.

As previsões das primeiras 24 horas vêm de `artefatos/previsoes_teste.csv`. As da estadia inteira vêm da seção 4.1. Nos dois casos o teste é o mesmo, com os mesmos pacientes.

**AUC e R² são escalas diferentes.** A AUC mede só se o modelo ordena bem os pacientes.

In [ ]:
previsoes_24h = pd.read_csv(PREVISOES_24H)
previsoes_estadia = pd.read_csv(SAIDA / "previsoes_estadia_completa.csv")
assert len(previsoes_24h) == len(previsoes_estadia), "testes diferentes"
assert np.allclose(previsoes_24h["los_dias"], previsoes_estadia["los_dias"]), "alvos diferentes"

linhas = []
for cenario, dados in [("primeiras 24 h", previsoes_24h), ("estadia inteira", previsoes_estadia)]:
    for corte in CORTES_DIAS:
        real = (dados["los_dias"] > corte).astype(int)
        for modelo in ["Ridge", "Lasso", "Random Forest", "XGBoost"]:
            nota = dados[modelo]
            linhas.append({
                "cenario": cenario,
                "corte_dias": corte,
                "modelo": modelo,
                "pct_longas": round(float(real.mean()) * 100, 1),
                "AUC": round(float(roc_auc_score(real, nota)), 3),
                "acuracia_balanceada": round(float(balanced_accuracy_score(real, (nota > corte).astype(int))), 3),
            })
auc = pd.DataFrame(linhas)
auc.to_csv(SAIDA / "auc_estadia_longa.csv", index=False, encoding="utf-8-sig")
display(auc[auc["modelo"] == "XGBoost"].reset_index(drop=True))

## 7. Como ler

- **R²** responde "quantos dias". **AUC** responde "este paciente é mais longo que o outro?". Não se comparam.
- Com as **primeiras 24 horas**, o R² é perto de zero e a AUC fica perto de 0,7.
- Com a **estadia inteira**, as métricas sobem porque o modelo vê valores extremos que só aparecem durante a permanência. Isso não é previsão na entrada da UTI.
- Ao citar a AUC da estadia inteira no TCC, diga que ela usa informação de toda a internação.